In [ ]:
import numpy as np
import mne
import os
import pywt
import scipy.io as sio
from scipy import signal
from scipy.signal import hilbert
from matplotlib.ticker import ScalarFormatter
from statsmodels.tsa.stattools import acf
from scipy.ndimage import gaussian_filter1d


from NeuralFieldManifold.models import AR, TAR
from NeuralFieldManifold.embedders import embed
from NeuralFieldManifold.pub_utils import *

# capture future warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

set_pub_style()

import numpy as np
from joblib import Parallel, delayed
from tqdm.auto import tqdm
import plotly.graph_objects as go
from NeuralFieldManifold.fits.two_torus import two_torus_fit, plot_two_torus_fit

In [ ]:
xs, time, Fs = np.load("data/monkey_15_min.npz").values()
xs_rescaled = (xs - np.min(xs)) / (np.max(xs) - np.min(xs)) * 2 - 1

In [ ]:
def sweep_fits(windows, lam=0.1, n_jobs=-1):
    fits = Parallel(n_jobs=n_jobs)(
        delayed(two_torus_fit)(w['points'], lam=lam)
        for w in windows
    )
    return [{**w, **f} for w, f in zip(windows, fits)]

def make_windows(xs, Fs, embedding_dim, tau, starts_sec, window_sec=2):
    window_samples = int(window_sec * Fs)
    windows = []
    for t in starts_sec:
        start = int(t * Fs)
        segment = xs[start:start + window_samples]
        pts = np.asarray(embed(segment, embedding_dim, tau))
        windows.append({
            'points': pts,
            'start_sec': t,
            'end_sec': t + window_sec,
        })
    return windows

In [ ]:
def generate_bad_ar_data(n_samples=600, ar_order=2, seed=None):
    """AR process with only real poles (no oscillatory modes), driven by white noise."""
    rng = np.random.default_rng(seed)

    if ar_order == 2:
        # two real stable poles — no oscillation
        p1, p2 = 0.6, -0.4
        a1 = p1 + p2        # 0.2
        a2 = -(p1 * p2)     # 0.24
        coeffs = np.array([a1, a2])
    elif ar_order == 4:
        # four real stable poles — no oscillation
        poles = np.array([0.55, -0.45, 0.35, -0.25])
        poly = np.polynomial.polynomial.polyfromroots(poles)
        # polyfromroots gives [c0, c1, ..., cn] for c0 + c1*z + ...
        # AR coeffs are -poly[:-1][::-1] / poly[-1] but poly[-1]=1 for monic
        # Actually for z^4 - a1*z^3 - ... = (z-p1)(z-p2)(z-p3)(z-p4)
        # easier: use np.poly which gives [1, -a1, -a2, ...] descending
        char_poly = np.poly(poles)  # [1, -(sum poles), ..., prod poles]
        coeffs = -char_poly[1:]     # AR coefficients
    else:
        raise ValueError("ar_order must be 2 or 4")

    x = np.zeros(n_samples, dtype=np.float32)
    x[:ar_order] = rng.standard_normal(ar_order).astype(np.float32)
    for i in range(ar_order, n_samples):
        ar_pred = sum(coeffs[j] * x[i - j - 1] for j in range(ar_order))
        x[i] = ar_pred + rng.standard_normal()
    return x


def envelope_normalize(x):
    return (x - np.min(x)) / (np.max(x) - np.min(x)) * 2 - 1

In [ ]:
window_size = 5
optimal_tau = 30
max_start = (len(xs_rescaled) / Fs) - window_size
sampled_starts = np.sort(np.random.uniform(0, max_start, 1000))

time_sweep_windows = make_windows(
    xs_rescaled,
    Fs=Fs,
    embedding_dim=3,
    tau=optimal_tau,
    starts_sec=sampled_starts.tolist(),
    window_sec=window_size,
)
torus_sweep = sweep_fits(time_sweep_windows)


In [ ]:
r_squared_vals = np.array([f['r_squared'] for f in torus_sweep])
r_squared_weights = np.ones_like(r_squared_vals) / len(r_squared_vals)

fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(r_squared_vals, bins=30, weights=r_squared_weights, color='black', alpha=0.5)
ax.axvline(r_squared_vals.mean(), color=color_alt, linewidth=2)
prettify(ax, xlabel="R²", ylabel="Probability", add_legend=False)
plt.tight_layout()
plt.show()


In [ ]:
torus_score_vals = np.array([f['frac_inside'] for f in torus_sweep])
torus_score_weights = np.ones_like(torus_score_vals) / len(torus_score_vals)

fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(torus_score_vals, bins=30, weights=torus_score_weights, color='black', alpha=0.5)
ax.axvline(torus_score_vals.mean(), color=color_alt, linewidth=2)
prettify(ax, xlabel="Torus Score", ylabel="Probability", add_legend=False)
plt.tight_layout()
plt.show()


In [ ]:
mean_error_vals = np.array([f['mean_error'] for f in torus_sweep])
mean_error_weights = np.ones_like(mean_error_vals) / len(mean_error_vals)

fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(mean_error_vals, bins=30, weights=mean_error_weights, color='black', alpha=0.5)
ax.axvline(mean_error_vals.mean(), color=color_alt, linewidth=2)
prettify(ax, xlabel="Torus Error", ylabel="Probability", add_legend=False)
plt.tight_layout()
plt.show()
